## 1. 기본 패키지 불러오기
ML feature process에 필요한 패키지를 불러옵니다.  

In [1]:
# 이 셀은 ML 산출물 생성에 필요한 패키지만 불러옵니다.
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import polars as pl

warnings.filterwarnings("ignore")

print("polars:", pl.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


polars: 0.20.31
pandas: 2.0.3
numpy: 1.25.2


## 2. 팀 공통 Google Drive 경로 설정
02번 preprocessing에서 생성한 `clean_base.parquet`을 읽고, ML용 산출물을 저장할 위치를 지정합니다.

In [2]:
# ============================================================
# TEAM COLAB PATH SETUP
# ============================================================
# 이 노트북은 Colab + Google Drive 실행을 기본으로 합니다.
#
# Drive 기준 폴더 구조:
# /content/drive/MyDrive/GraphAML
# └── data
#     ├── code       # 전처리 노트북
#     ├── raw        # 원본 데이터
#     ├── processed  # clean base 및 ML 산출물
#     └── graph      # GNN graph 산출물
#
# GitHub repo 기준:
# Graph_AML/
# └── data/
#     ├── raw/
#     ├── processed/
#     └── graph/
#
# csv/parquet 산출물은 GitHub PR에 포함하지 않고 Drive에 보관합니다.
# 폴더명이 다를 경우 GRAPHAML_DIR / GRAPHAML_RAW_DIR / GRAPHAML_RAW_DATA_PATH
# 환경변수로 직접 지정할 수 있습니다.
# ============================================================

from pathlib import Path
import os

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive") if IN_COLAB else Path("C:/Users/gusqh/Downloads")

PROJECT_DIR_CANDIDATES = [
    Path(os.environ["GRAPHAML_DIR"]) if "GRAPHAML_DIR" in os.environ else None,
    MYDRIVE / "GraphAML",
    MYDRIVE / "Graph_AML",
    MYDRIVE / "Graph AML",
]
PROJECT_DIR_CANDIDATES = [p for p in PROJECT_DIR_CANDIDATES if p is not None]


def pick_project_dir(candidates):
    """Drive 위치 차이를 줄이기 위해 프로젝트 폴더 후보를 자동 선택합니다."""
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    default = candidates[0] if candidates else (MYDRIVE / "GraphAML")
    Path(default).mkdir(parents=True, exist_ok=True)
    return Path(default)


BASE_DIR = pick_project_dir(PROJECT_DIR_CANDIDATES)
PROJECT_DIR = BASE_DIR
DATA_DIR = BASE_DIR / "data"

CODE_DIR = DATA_DIR / "code"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
GRAPH_DIR = DATA_DIR / "graph"

STEP01_DIR = PROCESSED_DIR / "step01_clean_base"
ML_DIR = PROCESSED_DIR / "ml_features"
GNN_DIR = GRAPH_DIR / "gnn_inputs"

# 기존 변수명과의 호환용 alias입니다. 실제 저장 위치는 새 data/ 구조입니다.
ML_DELIVERY_DIR = ML_DIR
GNN_DELIVERY_DIR = GNN_DIR
NOTEBOOK_DIR = CODE_DIR

for _p in [CODE_DIR, PROCESSED_DIR, GRAPH_DIR, STEP01_DIR, ML_DIR, GNN_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "BASE_DIR": BASE_DIR,
    "DATA_DIR": DATA_DIR,

    # 02번 산출물 위치 후보. 새 Drive 구조를 우선 사용합니다.
    "CLEAN_BASE_CANDIDATES": [
        STEP01_DIR / "clean_base.parquet",
        PROCESSED_DIR / "clean_base.parquet",
    ],

    "OUTPUT_DIR": ML_DIR,

    "RANDOM_SEED": 42,

    # 직접 재계산 검증은 비용 큼. 기본은 샘플 검증
    # 최종 제출 전 VALIDATION_MODE="full" 확인
    "SAMPLE_MODE": True,
    # 개발 중에는 샘플 검증을 쓰고, 최종 공유 전에는 full로 바꾸면 됩니다.
    "VALIDATION_MODE": "sample",  # "sample" 또는 "full"
    "MAX_VALIDATION_ROWS": 5000,

    "TRAIN_RATIO": 0.60,
    "VAL_RATIO": 0.20,
    "TEST_RATIO": 0.20,

    "WINDOWS": {
        "w1h": "1h",
        "w6h": "6h",
        "w1d": "1d",
        "w3d": "3d",
        "w7d": "7d",
    },

    # Small 데이터에서는 30d를 기본 feature로 만들지 않음
    "MAKE_OPTIONAL_30D": False,

    "COLUMN_CANDIDATES": {
        "timestamp": [
            "timestamp", "Timestamp", "time", "Time", "datetime", "DateTime"
        ],
        "label": [
            "is_laundering", "Is_Laundering", "label", "target", "y"
        ],
        "amount": [
            "amount", "Amount", "amount_paid", "Amount Paid", "paid_amount"
        ],

        # 02번 표준 컬럼명을 우선한다.
        "sender_account": [
            "sender_account_id", "sender_account", "from_account", "Account", "account",
            "orig_account", "source_account", "From Account"
        ],
        "receiver_account": [
            "receiver_account_id", "receiver_account", "to_account", "Account.1",
            "beneficiary_account", "dest_account", "destination_account", "To Account"
        ],
        "from_bank": [
            "sender_bank_id", "from_bank", "From Bank", "sender_bank", "source_bank"
        ],
        "to_bank": [
            "receiver_bank_id", "to_bank", "To Bank", "receiver_bank", "destination_bank"
        ],

        "payment_currency": [
            "payment_currency", "Payment Currency", "currency", "Currency"
        ],
        "receiving_currency": [
            "receiving_currency", "Receiving Currency", "received_currency"
        ],

        "payment_format": [
            "payment_format", "Payment Format", "payment_type", "Payment Type", "format"
        ],
        "tx_id": [
            "tx_id", "transaction_id", "TransactionID", "transaction_id_raw"
        ],
    },

    # 모델 입력 feature 점검용.
    # exact forbidden은 컬럼명 전체 일치만 금지하고
    # substring forbidden은 라벨/그래프 누수성 토큰만 검사한다.
    "EXACT_FORBIDDEN_MODEL_COLUMNS": {
        "label", "is_laundering", "tx_id", "transaction_id",
        "split", "timestamp", "datetime",
    },
    "SUBSTRING_FORBIDDEN_MODEL_TOKENS": {
        "laundering", "attempt_id", "pattern",
        "pagerank", "centrality", "degree",
    },
}

CONFIG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", CONFIG["BASE_DIR"])
print("DATA_DIR:", CONFIG["DATA_DIR"])
print("OUTPUT  :", CONFIG["OUTPUT_DIR"])
print("clean_base candidates:")
for p in CONFIG["CLEAN_BASE_CANDIDATES"]:
    print(" -", p, "exists=", Path(p).exists())


BASE_DIR: C:\Users\gusqh\Downloads\GraphAML
DATA_DIR: C:\Users\gusqh\Downloads\GraphAML\data
OUTPUT  : C:\Users\gusqh\Downloads\GraphAML\data\processed\ml_features
clean_base candidates:
 - C:\Users\gusqh\Downloads\GraphAML\data\processed\step01_clean_base\clean_base.parquet exists= True
 - C:\Users\gusqh\Downloads\GraphAML\data\processed\clean_base.parquet exists= True


## 3. 보조 함수 준비
split, feature catalog, leakage check 등에 반복해서 쓰는 함수입니다. 산출물의 신뢰도를 높이기 위한 검증 보조 코드입니다.

In [3]:
# ============================================================
# 보조 함수
# ============================================================

MEMORY_RECORDS = []

# 후보 경로 중 실제 존재하는 파일을 선택합니다.
def resolve_existing_path(candidates, name="input"):
    for path in candidates:
        if Path(path).exists():
            print("{} path: {}".format(name, path))
            return Path(path)

    msg = "{} 파일을 찾지 못했습니다. 확인한 후보 경로:\n{}".format(
        name,
        "\n".join([str(p) for p in candidates])
    )
    raise FileNotFoundError(msg)


def get_memory_mb():
    try:
        import psutil
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / 1024 / 1024
    except Exception:
        return np.nan


# 주요 단계별 메모리 사용량을 리포트용으로 기록합니다.
def record_memory(step, df=None):
    row_count = None
    col_count = None

    if df is not None:
        row_count = df.height
        col_count = len(df.columns)

    MEMORY_RECORDS.append({
        "step": step,
        "row_count": row_count,
        "col_count": col_count,
        "memory_mb": get_memory_mb(),
        "checked_at": pd.Timestamp.now().isoformat(),
    })

    print("[MEM]", step, "| rows=", row_count, "| cols=", col_count, "| mb=", round(get_memory_mb(), 2))


# 누적된 메모리 기록을 CSV로 저장합니다.
def save_memory_profile():
    path = CONFIG["OUTPUT_DIR"] / "memory_profile.csv"
    pd.DataFrame(MEMORY_RECORDS).to_csv(path, index=False, encoding="utf-8-sig")
    print("saved:", path)


def resolve_column(df, logical_name, required=True):
    candidates = CONFIG["COLUMN_CANDIDATES"].get(logical_name, [])
    existing = set(df.columns)

    for c in candidates:
        if c in existing:
            return c

    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]

    if required:
        raise ValueError(
            "필수 컬럼을 찾지 못했습니다. logical_name={}, candidates={}, columns={}".format(
                logical_name, candidates, df.columns
            )
        )

    return None


def safe_log1p_expr(col):
    return (
        pl.when(pl.col(col).is_null() | (pl.col(col) < 0))
        .then(None)
        .otherwise(pl.col(col).log1p())
    )


def parse_window_to_ns(window_str):
    unit = window_str[-1]
    value = int(window_str[:-1])

    if unit == "h":
        seconds = value * 3600
    elif unit == "d":
        seconds = value * 24 * 3600
    elif unit == "m":
        seconds = value * 60
    else:
        raise ValueError("지원하지 않는 window입니다: {}".format(window_str))

    return int(seconds * 1_000_000_000)


def series_datetime_to_ns(series):
    return series.to_numpy().astype("datetime64[ns]").astype("int64")


def safe_schema_get(schema, col, default="unknown"):
    return str(schema[col]) if col in schema else default


def print_shape(name, df):
    print("{}: rows={:,}, cols={:,}".format(name, df.height, len(df.columns)))


## 4. clean_base 로드
02번에서 만든 공통 기준 데이터를 읽습니다. 이 단계부터는 원본 CSV가 아니라 clean base를 기준으로 ML 파일을 만듭니다.

In [4]:
# ============================================================
# clean_base 로드 및 기본 컬럼 정리
# ============================================================

# 01번 산출물 후보 중 실제 존재하는 clean_base를 읽습니다.
clean_path = resolve_existing_path(CONFIG["CLEAN_BASE_CANDIDATES"], name="clean_base")

df_raw = pl.read_parquet(clean_path)
record_memory("load_clean_base", df_raw)
print_shape("df_raw", df_raw)

COL = {
    "timestamp": resolve_column(df_raw, "timestamp", required=True),
    "label": resolve_column(df_raw, "label", required=True),
    "amount": resolve_column(df_raw, "amount", required=True),
    "sender_account": resolve_column(df_raw, "sender_account", required=True),
    "receiver_account": resolve_column(df_raw, "receiver_account", required=True),
    "from_bank": resolve_column(df_raw, "from_bank", required=False),
    "to_bank": resolve_column(df_raw, "to_bank", required=False),
    "payment_currency": resolve_column(df_raw, "payment_currency", required=False),
    "receiving_currency": resolve_column(df_raw, "receiving_currency", required=False),
    "payment_format": resolve_column(df_raw, "payment_format", required=False),
    "tx_id": resolve_column(df_raw, "tx_id", required=False),
}

print("매칭된 컬럼")
for k, v in COL.items():
    print("-", k, ":", v)

# clone 없이 바로 처리
df = df_raw

# tx_id 없으면 row 순서 기준 생성
if COL["tx_id"] is None:
    df = df.with_row_index("tx_id")
    COL["tx_id"] = "tx_id"

# 은행 / 통화 / 결제방식 컬럼 없으면 unknown
for key in ["from_bank", "to_bank", "payment_currency", "receiving_currency", "payment_format"]:
    if COL[key] is None:
        # split 라벨을 원본 row에 붙여 이후 산출물에서 공통으로 사용합니다.
        df = df.with_columns(pl.lit("unknown").alias(key))
        COL[key] = key

# split 라벨을 원본 row에 붙여 이후 산출물에서 공통으로 사용합니다.
df = df.with_columns([
    pl.col(COL["tx_id"]).cast(pl.Utf8).alias("tx_id"),
    pl.col(COL["timestamp"]).cast(pl.Datetime).alias("timestamp"),
    pl.col(COL["label"]).cast(pl.Int64).alias("label"),
    pl.col(COL["amount"]).cast(pl.Float64).alias("amount"),

    pl.col(COL["sender_account"]).cast(pl.Utf8).alias("sender_account"),
    pl.col(COL["receiver_account"]).cast(pl.Utf8).alias("receiver_account"),

    pl.col(COL["from_bank"]).cast(pl.Utf8).fill_null("unknown").alias("from_bank"),
    pl.col(COL["to_bank"]).cast(pl.Utf8).fill_null("unknown").alias("to_bank"),
    pl.col(COL["payment_currency"]).cast(pl.Utf8).fill_null("unknown").alias("payment_currency"),
    pl.col(COL["receiving_currency"]).cast(pl.Utf8).fill_null("unknown").alias("receiving_currency"),
    pl.col(COL["payment_format"]).cast(pl.Utf8).fill_null("unknown").alias("payment_format"),
])

# 02번에서도 label을 한 번 더 확인해 잘못된 입력을 막습니다.
bad_label_df = df.filter(
    pl.col("label").is_null() |
    (~pl.col("label").is_in([0, 1]))
)

invalid_label_count = bad_label_df.height
null_timestamp_count = df.filter(pl.col("timestamp").is_null()).height
null_amount_count = df.filter(pl.col("amount").is_null()).height

print("invalid_label_count:", invalid_label_count)
print("null_timestamp_count:", null_timestamp_count)
print("null_amount_count:", null_amount_count)

if invalid_label_count > 0:
    bad_label_path = CONFIG["OUTPUT_DIR"] / "step02_bad_label_examples.csv"
    bad_label_df.head(50).write_csv(bad_label_path)
    raise ValueError(
        "label이 null 또는 0/1 외 값입니다. rows={}, examples={}".format(
            invalid_label_count,
            bad_label_path,
        )
    )

if null_timestamp_count > 0:
    raise ValueError("timestamp null row가 있습니다.")
if null_amount_count > 0:
    raise ValueError("amount null row가 있습니다.")

df = df.sort(["timestamp", "tx_id"])

def validate_tx_id_unique_in_step02(df):
    n_rows = df.height
    n_unique = df.select(pl.col("tx_id").n_unique()).item()

    if n_rows != n_unique:
        raise AssertionError(
            "tx_id 중복 발생: rows={}, unique_tx_id={}".format(n_rows, n_unique)
        )

    print("tx_id unique check: PASS | rows={} unique_tx_id={}".format(n_rows, n_unique))


validate_tx_id_unique_in_step02(df)

record_memory("canonical_schema_ready", df)

df.select([
    "tx_id", "timestamp", "label", "amount",
    "sender_account", "receiver_account",
    "from_bank", "to_bank", "payment_currency", "receiving_currency", "payment_format"
]).head(5)


clean_base path: C:\Users\gusqh\Downloads\GraphAML\data\processed\step01_clean_base\clean_base.parquet
[MEM] load_clean_base | rows= 5078345 | cols= 23 | mb= 1888.68
df_raw: rows=5,078,345, cols=23
매칭된 컬럼
- timestamp : timestamp
- label : is_laundering
- amount : amount
- sender_account : sender_account_id
- receiver_account : receiver_account_id
- from_bank : sender_bank_id
- to_bank : receiver_bank_id
- payment_currency : Payment Currency
- receiving_currency : Receiving Currency
- payment_format : Payment Format
- tx_id : tx_id
invalid_label_count: 0
null_timestamp_count: 0
null_amount_count: 0
tx_id unique check: PASS | rows=5078345 unique_tx_id=5078345
[MEM] canonical_schema_ready | rows= 5078345 | cols= 31 | mb= 983.05


tx_id,timestamp,label,amount,sender_account,receiver_account,from_bank,to_bank,payment_currency,receiving_currency,payment_format
str,datetime[μs],i64,f64,str,str,str,str,str,str,str
"""0""",2022-09-01 00:00:00,0,14675.57,"""03209|8000F4670""","""03209|8000F4670""","""03209""","""03209""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""1""",2022-09-01 00:00:00,0,897.37,"""01420|8005DFEB0""","""01420|8005DFEB0""","""01420""","""01420""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""10""",2022-09-01 00:00:00,0,12971.84,"""03284|800146EE0""","""03284|800146EE0""","""03284""","""03284""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""100""",2022-09-01 00:00:00,0,15.69,"""01601|80056E5C0""","""01601|80056E5C0""","""01601""","""01601""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""1000""",2022-09-01 00:00:00,0,924649.12,"""019474|803ACC410""","""019474|803ACC410""","""019474""","""019474""","""US Dollar""","""US Dollar""","""Reinvestment"""


## 5. 시간순 Train/Validation/Test split
계획서 기준에 따라 Timestamp 기준 시간순 60/20/20 split을 만듭니다. 동일 Timestamp가 서로 다른 split에 나뉘지 않도록 확인하는 것이 핵심입니다.

In [5]:
# ============================================================
# timestamp 기준 60/20/20 split
# 같은 timestamp는 같은 split에 들어가도록 한다.
# ============================================================

# timestamp 기준으로 train/val/test를 시간순 분할합니다.
def make_time_split(df):
    n = df.height

    train_target = n * CONFIG["TRAIN_RATIO"]
    val_target = n * (CONFIG["TRAIN_RATIO"] + CONFIG["VAL_RATIO"])

    ts_counts = (
        df.group_by("timestamp")
        .agg(pl.len().alias("n_rows"))
        .sort("timestamp")
        .with_columns(pl.col("n_rows").cum_sum().alias("cum_rows"))
    )

    cum = ts_counts["cum_rows"].to_numpy()
    timestamps = ts_counts["timestamp"].to_list()

    if len(timestamps) < 3:
        raise ValueError("timestamp 종류가 너무 적어 train/val/test split을 만들 수 없습니다.")

    # timestamp 그룹을 쪼개지 않도록 split 경계를 조정합니다.
    def choose_boundary(target):
        idx = int(np.searchsorted(cum, target, side="left"))

        if idx <= 0:
            return 1
        if idx >= len(cum):
            return len(cum) - 1

        prev_diff = abs(cum[idx - 1] - target)
        curr_diff = abs(cum[idx] - target)

        if prev_diff <= curr_diff:
            return idx
        return idx + 1

    train_group_end = choose_boundary(train_target)
    val_group_end = choose_boundary(val_target)

    train_group_end = max(1, min(train_group_end, len(timestamps) - 2))
    val_group_end = max(train_group_end + 1, min(val_group_end, len(timestamps) - 1))

    train_last_ts = timestamps[train_group_end - 1]
    val_last_ts = timestamps[val_group_end - 1]

    print("train_last_ts:", train_last_ts)
    print("val_last_ts  :", val_last_ts)

    return df.with_columns(
        pl.when(pl.col("timestamp") <= train_last_ts)
        .then(pl.lit("train"))
        .when(pl.col("timestamp") <= val_last_ts)
        .then(pl.lit("val"))
        .otherwise(pl.lit("test"))
        .alias("split")
    )


df = make_time_split(df)
record_memory("time_split_created", df)

df.group_by("split").agg([
    pl.len().alias("row_count"),
    pl.col("timestamp").min().alias("timestamp_min"),
    pl.col("timestamp").max().alias("timestamp_max"),
    pl.col("label").sum().alias("label_1_count"),
]).sort("timestamp_min")


train_last_ts: 2022-09-06 13:35:00
val_last_ts  : 2022-09-08 16:12:00
[MEM] time_split_created | rows= 5078345 | cols= 32 | mb= 749.14


split,row_count,timestamp_min,timestamp_max,label_1_count
str,u32,datetime[μs],datetime[μs],i64
"""train""",3046861,2022-09-01 00:00:00,2022-09-06 13:35:00,2297
"""val""",1015920,2022-09-06 13:36:00,2022-09-08 16:12:00,1083
"""test""",1015564,2022-09-08 16:13:00,2022-09-18 16:18:00,1797


## 6. baseline feature 구성
가장 기본적인 ML 입력 피처를 구성합니다. 라벨이나 라벨과 직접 연결될 수 있는 정보는 모델 입력에서 제외합니다.

In [6]:
# ============================================================
# split_summary.csv
# ============================================================

# split별 row 수, 라벨 분포, timestamp 범위를 요약합니다.
def make_split_summary(df):
    summary = (
        df.group_by("split")
        .agg([
            pl.len().alias("row_count"),
            (pl.col("label") == 0).sum().alias("label_0_count"),
            (pl.col("label") == 1).sum().alias("label_1_count"),
            pl.col("label").mean().alias("label_1_ratio"),
            pl.col("timestamp").min().alias("timestamp_min"),
            pl.col("timestamp").max().alias("timestamp_max"),
        ])
    )

    order = pl.DataFrame({
        "split": ["train", "val", "test"],
        "split_order": [0, 1, 2],
    })

    return (
        summary.join(order, on="split", how="left")
        .sort("split_order")
        .drop("split_order")
    )


# split 결과는 별도 CSV로 저장합니다.
split_summary = make_split_summary(df)
split_summary_path = CONFIG["OUTPUT_DIR"] / "split_summary.csv"
split_summary.write_csv(split_summary_path)

print("saved:", split_summary_path)
split_summary


saved: C:\Users\gusqh\Downloads\GraphAML\data\processed\ml_features\split_summary.csv


split,row_count,label_0_count,label_1_count,label_1_ratio,timestamp_min,timestamp_max
str,u32,u32,u32,f64,datetime[μs],datetime[μs]
"""train""",3046861,3044564,2297,0.000754,2022-09-01 00:00:00,2022-09-06 13:35:00
"""val""",1015920,1014837,1083,0.001066,2022-09-06 13:36:00,2022-09-08 16:12:00
"""test""",1015564,1013767,1797,0.001769,2022-09-08 16:13:00,2022-09-18 16:18:00


## 7. time feature 포함 버전 생성
시간대, 요일 등 시간 기반 피처가 성능에 도움이 되는지 비교하기 위한 버전입니다. 최종 사용 여부는 baseline 결과를 보고 판단합니다.

In [7]:
# ============================================================
# ML-00: 현재 거래에서 바로 알 수 있는 feature
# ============================================================

# 현재 거래에서 바로 알 수 있는 baseline feature를 만듭니다.
def add_ml00_features(df):
    return df.with_columns([
        pl.col("amount").alias("amount__current__value"),
        safe_log1p_expr("amount").alias("amount__current__log1p"),

        pl.col("timestamp").dt.hour().cast(pl.Int16).alias("time__row__hour"),
        pl.col("timestamp").dt.weekday().cast(pl.Int16).alias("time__row__dayofweek"),
        (pl.col("timestamp").dt.weekday() >= 6).cast(pl.Int8).alias("time__row__is_weekend"),
    ])


df_ml00_base = add_ml00_features(df)
record_memory("ml00_base_features_created", df_ml00_base)

df_ml00_base.select([
    "tx_id", "timestamp", "split", "label",
    "amount__current__value",
    "amount__current__log1p",
    "time__row__hour",
    "time__row__dayofweek",
    "time__row__is_weekend",
    "from_bank", "to_bank", "payment_currency", "receiving_currency", "payment_format"
]).head(5)


[MEM] ml00_base_features_created | rows= 5078345 | cols= 37 | mb= 1036.79


tx_id,timestamp,split,label,amount__current__value,amount__current__log1p,time__row__hour,time__row__dayofweek,time__row__is_weekend,from_bank,to_bank,payment_currency,receiving_currency,payment_format
str,datetime[μs],str,i64,f64,f64,i16,i16,i8,str,str,str,str,str
"""0""",2022-09-01 00:00:00,"""train""",0,14675.57,9.594008,0,4,0,"""03209""","""03209""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""1""",2022-09-01 00:00:00,"""train""",0,897.37,6.800582,0,4,0,"""01420""","""01420""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""10""",2022-09-01 00:00:00,"""train""",0,12971.84,9.470613,0,4,0,"""03284""","""03284""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""100""",2022-09-01 00:00:00,"""train""",0,15.69,2.81481,0,4,0,"""01601""","""01601""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""1000""",2022-09-01 00:00:00,"""train""",0,924649.12,13.737171,0,4,0,"""019474""","""019474""","""US Dollar""","""US Dollar""","""Reinvestment"""


## 8. no-time 버전 생성
시간 피처를 제외한 비교군입니다. “시간 피처가 정말 필요한가?”를 실험적으로 확인하기 위한 파일입니다.

In [8]:
# ============================================================
# 범주형 인코딩
# fit은 train 데이터만 사용한다.
# val/test 신규 category는 -1로 둔다.
# ============================================================

CATEGORICAL_RAW_COLS = [
    "from_bank",
    "to_bank",
    "payment_currency",
    "receiving_currency",
    "payment_format",
]

CAT_FEATURE_MAP = {
    "from_bank": "cat__from_bank__code",
    "to_bank": "cat__to_bank__code",
    "payment_currency": "cat__payment_currency__code",
    "receiving_currency": "cat__receiving_currency__code",
    "payment_format": "cat__payment_format__code",
}


def fit_category_maps_train_only(df, cols):
    train_df = df.filter(pl.col("split") == "train")
    maps = {}

    for col in cols:
        values = (
            train_df
            .select(pl.col(col).cast(pl.Utf8).fill_null("unknown"))
            .unique()
            .sort(col)
            .to_series()
            .to_list()
        )

        maps[col] = {str(v): i for i, v in enumerate(values)}
        print("[fit]", col, "n=", len(maps[col]))

    return maps


# train에서 만든 mapping으로 val/test를 변환합니다.
def apply_category_maps(df, cat_maps):
    out = df

    for raw_col, mapping in cat_maps.items():
        feature_col = CAT_FEATURE_MAP[raw_col]

        map_df = pl.DataFrame({
            raw_col: list(mapping.keys()),
            feature_col: list(mapping.values()),
        }).with_columns([
            pl.col(raw_col).cast(pl.Utf8),
            pl.col(feature_col).cast(pl.Int32),
        ])

        out = (
            out.with_columns(pl.col(raw_col).cast(pl.Utf8).fill_null("unknown"))
            .join(map_df, on=raw_col, how="left")
            .with_columns(pl.col(feature_col).fill_null(-1).cast(pl.Int32))
        )

    return out


# category encoder는 train split만 기준으로 fit합니다.
category_maps = fit_category_maps_train_only(df_ml00_base, CATEGORICAL_RAW_COLS)
df_ml00_base = apply_category_maps(df_ml00_base, category_maps)

cat_map_records = []
for raw_col, mapping in category_maps.items():
    for category, code in mapping.items():
        cat_map_records.append({
            "raw_column": raw_col,
            "feature_column": CAT_FEATURE_MAP[raw_col],
            "category": category,
            "code": code,
            "fit_scope": "train_only",
        })

category_map_path = CONFIG["OUTPUT_DIR"] / "category_mapping_train_only.csv"
pd.DataFrame(cat_map_records).to_csv(category_map_path, index=False, encoding="utf-8-sig")
print("saved:", category_map_path)

record_memory("categorical_encoded_train_only", df_ml00_base)

df_ml00_base.select([
    "split",
    "from_bank", "cat__from_bank__code",
    "to_bank", "cat__to_bank__code",
    "payment_currency", "cat__payment_currency__code",
    "receiving_currency", "cat__receiving_currency__code",
    "payment_format", "cat__payment_format__code",
]).head(5)


[fit] from_bank n= 30528
[fit] to_bank n= 15850
[fit] payment_currency n= 15
[fit] receiving_currency n= 15
[fit] payment_format n= 7
saved: C:\Users\gusqh\Downloads\GraphAML\data\processed\ml_features\category_mapping_train_only.csv
[MEM] categorical_encoded_train_only | rows= 5078345 | cols= 42 | mb= 1723.24


split,from_bank,cat__from_bank__code,to_bank,cat__to_bank__code,payment_currency,cat__payment_currency__code,receiving_currency,cat__receiving_currency__code,payment_format,cat__payment_format__code
str,str,i32,str,i32,str,i32,str,i32,str,i32
"""train""","""03209""",8038,"""03209""",5027,"""US Dollar""",12,"""US Dollar""",12,"""Reinvestment""",5
"""train""","""01420""",972,"""01420""",972,"""US Dollar""",12,"""US Dollar""",12,"""Reinvestment""",5
"""train""","""03284""",12390,"""03284""",7248,"""US Dollar""",12,"""US Dollar""",12,"""Reinvestment""",5
"""train""","""01601""",1131,"""01601""",1131,"""US Dollar""",12,"""US Dollar""",12,"""Reinvestment""",5
"""train""","""019474""",1185,"""019474""",1185,"""US Dollar""",12,"""US Dollar""",12,"""Reinvestment""",5


## 9. Stage 0 safe history feature 생성
누수 방지를 가장 우선으로 두고, `timestamp < current_timestamp` 조건의 과거 거래만 사용합니다. 현재 row와 동일 timestamp 거래는 과거로 보지 않습니다.

In [ ]:
# 누수 방지 기준이 가장 중요한 history feature
# ============================================================
# Stage 0 history feature
# 조건: current_timestamp - window <= past_timestamp < current_timestamp
# 동일 timestamp 거래는 과거로 보지 않는다.
# ============================================================

def compute_entity_history_features(
    df,
    entity_col,
    amount_col,
    timestamp_col,
    direction_label,
    entity_scope_label,
    windows,
):
    work = df.select(["tx_id", entity_col, amount_col, timestamp_col]).sort(
        [entity_col, timestamp_col, "tx_id"]
    )

    tx_ids = work["tx_id"].to_list()
    entities = work[entity_col].to_list()
    amounts = work[amount_col].to_numpy()
    ts_ns = series_datetime_to_ns(work[timestamp_col])

    n = work.height
    result = {"tx_id": tx_ids}

    for w_name in windows.keys():
        result["timehist__{}__{}__tx_count__count__{}".format(
            entity_scope_label, direction_label, w_name
        )] = np.zeros(n, dtype=np.int64)

        result["timehist__{}__{}__amount__sum__{}".format(
            entity_scope_label, direction_label, w_name
        )] = np.zeros(n, dtype=np.float64)

        result["timehist__{}__{}__amount__mean__{}".format(
            entity_scope_label, direction_label, w_name
        )] = np.zeros(n, dtype=np.float64)

    # timestamp라는 토큰은 금지 컬럼 검사에서 걸리므로 피처명에 넣지 않는다.
    recency_col = "recency__{}__{}__seconds_since_last".format(
        entity_scope_label, direction_label
    )
    first_col = "flag__{}__{}__is_first_tx".format(
        entity_scope_label, direction_label
    )

    result[recency_col] = np.full(n, np.nan, dtype=np.float64)
    result[first_col] = np.zeros(n, dtype=np.int8)

    entity_to_indices = {}
    for i, ent in enumerate(entities):
        entity_to_indices.setdefault(ent, []).append(i)

    for ent, idx_list in entity_to_indices.items():
        idx_arr = np.array(idx_list, dtype=np.int64)
        ent_ts = ts_ns[idx_arr]
        ent_amount = amounts[idx_arr].astype(float)
        prefix_amount = np.concatenate([[0.0], np.cumsum(ent_amount)])

        for local_pos, global_idx in enumerate(idx_arr):
            current_ts = ent_ts[local_pos]

            # side='left'라서 같은 timestamp 거래는 제외된다.
            end = np.searchsorted(ent_ts, current_ts, side="left")

            if end == 0:
                result[first_col][global_idx] = 1
                result[recency_col][global_idx] = np.nan
            else:
                last_ts = ent_ts[end - 1]
                result[recency_col][global_idx] = (current_ts - last_ts) / 1_000_000_000

            for w_name, w_str in windows.items():
                window_ns = parse_window_to_ns(w_str)
                start_ts = current_ts - window_ns
                start = np.searchsorted(ent_ts, start_ts, side="left")

                count = end - start
                amount_sum = prefix_amount[end] - prefix_amount[start]
                amount_mean = amount_sum / count if count > 0 else 0.0

                count_col = "timehist__{}__{}__tx_count__count__{}".format(
                    entity_scope_label, direction_label, w_name
                )
                sum_col = "timehist__{}__{}__amount__sum__{}".format(
                    entity_scope_label, direction_label, w_name
                )
                mean_col = "timehist__{}__{}__amount__mean__{}".format(
                    entity_scope_label, direction_label, w_name
                )

                result[count_col][global_idx] = count
                result[sum_col][global_idx] = amount_sum
                result[mean_col][global_idx] = amount_mean

    return pl.DataFrame(result)


windows = dict(CONFIG["WINDOWS"])
if CONFIG["MAKE_OPTIONAL_30D"]:
    windows["optional_30d"] = "30d"

print("windows:", windows)

sender_hist = compute_entity_history_features(
    df=df_ml00_base,
    entity_col="sender_account",
    amount_col="amount",
    timestamp_col="timestamp",
    direction_label="out",
    entity_scope_label="sender",
    windows=windows,
)
record_memory("sender_history_created", sender_hist)

receiver_hist = compute_entity_history_features(
    df=df_ml00_base,
    entity_col="receiver_account",
    amount_col="amount",
    timestamp_col="timestamp",
    direction_label="in",
    entity_scope_label="receiver",
    windows=windows,
)
record_memory("receiver_history_created", receiver_hist)

print_shape("sender_hist", sender_hist)
print_shape("receiver_hist", receiver_hist)


windows: {'w1h': '1h', 'w6h': '6h', 'w1d': '1d', 'w3d': '3d', 'w7d': '7d'}


## 10. feature catalog 작성
단순 feature list가 아니라 각 피처의 목적, 입력 컬럼, 누수 방지 규칙, null policy를 기록합니다. 추후 피처 제거/추가 논의의 기준 문서입니다.

In [ ]:
# ============================================================
# ML-01 구성
# ============================================================

# baseline feature와 history feature를 tx_id 기준으로 결합합니다.
df_ml01_base = (
    df_ml00_base
    .join(sender_hist, on="tx_id", how="left")
    .join(receiver_hist, on="tx_id", how="left")
)

fill_zero_cols = [
    c for c in df_ml01_base.columns
    if c.startswith("timehist__") or c.startswith("flag__")
]

df_ml01_base = df_ml01_base.with_columns([
    pl.col(c).fill_null(0) for c in fill_zero_cols
])

record_memory("ml01_base_created", df_ml01_base)
print_shape("df_ml01_base", df_ml01_base)

history_cols_preview = [c for c in df_ml01_base.columns if c.startswith("timehist__")][:10]
recency_cols_preview = [c for c in df_ml01_base.columns if c.startswith("recency__")]

df_ml01_base.select(
    ["tx_id", "timestamp", "split", "label"] + history_cols_preview + recency_cols_preview
).head(5)


[MEM] ml01_base_created | rows= 5078345 | cols= 76 | mb= 4782.39
df_ml01_base: rows=5,078,345, cols=76


tx_id,timestamp,split,label,timehist__sender__out__tx_count__count__w1h,timehist__sender__out__amount__sum__w1h,timehist__sender__out__amount__mean__w1h,timehist__sender__out__tx_count__count__w6h,timehist__sender__out__amount__sum__w6h,timehist__sender__out__amount__mean__w6h,timehist__sender__out__tx_count__count__w1d,timehist__sender__out__amount__sum__w1d,timehist__sender__out__amount__mean__w1d,timehist__sender__out__tx_count__count__w3d,recency__sender__out__seconds_since_last,recency__receiver__in__seconds_since_last
str,datetime[μs],str,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,f64
"""0""",2022-09-01 00:00:00,"""train""",0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,NaN,NaN
"""1""",2022-09-01 00:00:00,"""train""",0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,NaN,NaN
"""10""",2022-09-01 00:00:00,"""train""",0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,NaN,NaN
"""100""",2022-09-01 00:00:00,"""train""",0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,NaN,NaN
"""1000""",2022-09-01 00:00:00,"""train""",0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,NaN,NaN


## 11. leakage check 실행
현재 row 포함, 동일 timestamp 포함, 미래 거래 참조, 라벨 유사 정보 포함 여부를 점검합니다. P0 문제가 있으면 이후 파일 생성을 중단하는 것이 원칙입니다.

In [ ]:
# ============================================================
# 모델 입력 feature 목록
# meta 컬럼은 parquet에는 남기되 학습 feature에서는 제외한다.
# ============================================================

META_COLS = [
    "tx_id",
    "timestamp",
    "split",
    "label",
    "sender_account",
    "receiver_account",
]

ML00_FEATURE_COLS = [
    "amount__current__value",
    "amount__current__log1p",
    "cat__from_bank__code",
    "cat__to_bank__code",
    "cat__payment_currency__code",
    "cat__receiving_currency__code",
    "cat__payment_format__code",
    "time__row__hour",
    "time__row__dayofweek",
    "time__row__is_weekend",
]

ML00_NO_TIME_FEATURE_COLS = [
    c for c in ML00_FEATURE_COLS
    if not c.startswith("time__")
]

ML01_HISTORY_FEATURE_COLS = [
    c for c in df_ml01_base.columns
    if c.startswith("timehist__") or c.startswith("recency__") or c.startswith("flag__")
]

ML01_FEATURE_COLS = ML00_FEATURE_COLS + ML01_HISTORY_FEATURE_COLS

print("ML00:", len(ML00_FEATURE_COLS))
print("ML00 no time:", len(ML00_NO_TIME_FEATURE_COLS))
print("ML01 history:", len(ML01_HISTORY_FEATURE_COLS))
print("ML01 total:", len(ML01_FEATURE_COLS))

for c in ML01_FEATURE_COLS:
    if c not in df_ml01_base.columns:
        raise ValueError("feature column missing: {}".format(c))


ML00: 10
ML00 no time: 7
ML01 history: 34
ML01 total: 44


## 12. split summary 저장
split별 row 수, 라벨 분포, Timestamp 범위를 저장합니다. 

In [ ]:
# ============================================================
# 최종 ML 데이터셋
# ============================================================

# DataFrame에 실제 존재하는 컬럼만 선택합니다.
def select_existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

META_KEEP_COLS = select_existing_cols(df_ml01_base, META_COLS)

ml_exp00 = df_ml00_base.select(META_KEEP_COLS + ML00_FEATURE_COLS)
ml_exp00_no_time = df_ml00_base.select(META_KEEP_COLS + ML00_NO_TIME_FEATURE_COLS)
ml_exp01 = df_ml01_base.select(META_KEEP_COLS + ML01_FEATURE_COLS)

# 이름을 더 명확하게 둔다. label은 y 분리용이고, split은 분할 유지용이다.
ml_exp00_Xy = df_ml00_base.select(["tx_id", "split", "label"] + ML00_FEATURE_COLS)
ml_exp00_no_time_Xy = df_ml00_base.select(["tx_id", "split", "label"] + ML00_NO_TIME_FEATURE_COLS)
ml_exp01_Xy = df_ml01_base.select(["tx_id", "split", "label"] + ML01_FEATURE_COLS)

record_memory("ml_exp00_ready", ml_exp00)
record_memory("ml_exp00_no_time_ready", ml_exp00_no_time)
record_memory("ml_exp01_ready", ml_exp01)

print_shape("ml_exp00", ml_exp00)
print_shape("ml_exp00_no_time", ml_exp00_no_time)
print_shape("ml_exp01", ml_exp01)
print_shape("ml_exp00_Xy", ml_exp00_Xy)
print_shape("ml_exp00_no_time_Xy", ml_exp00_no_time_Xy)
print_shape("ml_exp01_Xy", ml_exp01_Xy)


[MEM] ml_exp00_ready | rows= 5078345 | cols= 16 | mb= 4782.62
[MEM] ml_exp00_no_time_ready | rows= 5078345 | cols= 13 | mb= 4782.62
[MEM] ml_exp01_ready | rows= 5078345 | cols= 50 | mb= 4782.62
ml_exp00: rows=5,078,345, cols=16
ml_exp00_no_time: rows=5,078,345, cols=13
ml_exp01: rows=5,078,345, cols=50
ml_exp00_Xy: rows=5,078,345, cols=13
ml_exp00_no_time_Xy: rows=5,078,345, cols=10
ml_exp01_Xy: rows=5,078,345, cols=47


## 13. ML 산출물 저장
ML 실험에 사용할 parquet/csv 파일을 저장합니다. 산출물 이름은 기존 팀 공유 기준을 유지했습니다.

In [ ]:
# ============================================================
# feature_catalog.csv
# ============================================================

# feature별 역할, 누수 규칙, null 처리 정리합니다.
def make_feature_catalog():
    records = []

    def add_record(
        feature_name,
        stage,
        feature_family,
        entity_scope,
        direction,
        window,
        aggregation_function,
        input_columns,
        leakage_rule,
        null_policy,
        computational_cost,
        aml_typology,
        selection_status,
        notes="",
        correlation_group="",
    ):
        records.append({
            "feature_name": feature_name,
            "stage": stage,
            "feature_family": feature_family,
            "entity_scope": entity_scope,
            "direction": direction,
            "window": window,
            "aggregation_function": aggregation_function,
            "input_columns": input_columns,
            "leakage_rule": leakage_rule,
            "null_policy": null_policy,
            "computational_cost": computational_cost,
            "aml_typology": aml_typology,
            "selection_status": selection_status,
            "notes": notes,
            "correlation_group": correlation_group,
        })

    add_record(
        "amount__current__value", "ML-00", "amount", "transaction", "current", "none",
        "identity", "amount",
        "current row only",
        "source amount must be non-null",
        "low", "amount anomaly", "selected",
    )

    add_record(
        "amount__current__log1p", "ML-00", "amount", "transaction", "current", "none",
        "log1p", "amount",
        "current row only",
        "negative amount -> null",
        "low", "amount anomaly", "selected",
        "raw amount와 같이 실험 후 비교", "amount_scale"
    )

    for raw_col, feat_col in CAT_FEATURE_MAP.items():
        add_record(
            feat_col, "ML-00", "categorical", "transaction", "current", "none",
            "train_only_code", raw_col,
            "mapping is fitted on train split only",
            "unknown in val/test -> -1",
            "low", "institution/payment/currency pattern", "selected",
            "정수 코드 자체에 순서 의미는 없음",
            "categorical_{}".format(raw_col)
        )

    for feat, agg, note in [
        ("time__row__hour", "hour", "transaction hour"),
        ("time__row__dayofweek", "dayofweek", "weekday"),
        ("time__row__is_weekend", "is_weekend", "weekend flag"),
    ]:
        add_record(
            feat, "ML-00", "time", "transaction", "current", "none",
            agg, "timestamp",
            "derived from current timestamp only",
            "timestamp valid -> not null",
            "low", "temporal pattern", "selected",
            note, "time_current"
        )

    for w_name, w_str in windows.items():
        for entity_scope, direction in [("sender", "out"), ("receiver", "in")]:
            add_record(
                "timehist__{}__{}__tx_count__count__{}".format(entity_scope, direction, w_name),
                "ML-01", "timehist", entity_scope, direction, w_name,
                "count", "{},timestamp".format(entity_scope),
                "{} window; past_timestamp < current_timestamp".format(w_str),
                "no history -> 0",
                "medium", "velocity", "selected",
                "window count, not global count",
                "{}_{}_velocity".format(entity_scope, direction)
            )

            add_record(
                "timehist__{}__{}__amount__sum__{}".format(entity_scope, direction, w_name),
                "ML-01", "timehist", entity_scope, direction, w_name,
                "sum", "{},timestamp,amount".format(entity_scope),
                "{} window; past_timestamp < current_timestamp".format(w_str),
                "no history -> 0",
                "medium", "amount burst", "selected",
                "window sum, not global sum",
                "{}_{}_amount_window".format(entity_scope, direction)
            )

            add_record(
                "timehist__{}__{}__amount__mean__{}".format(entity_scope, direction, w_name),
                "ML-01", "timehist", entity_scope, direction, w_name,
                "mean", "{},timestamp,amount".format(entity_scope),
                "{} window; past_timestamp < current_timestamp".format(w_str),
                "zero count -> 0",
                "medium", "average behavior", "selected",
                "sum/count와 상관 가능성 있음. 여기서는 제거하지 않음",
                "{}_{}_amount_window".format(entity_scope, direction)
            )

    for entity_scope, direction in [("sender", "out"), ("receiver", "in")]:
        add_record(
            "recency__{}__{}__seconds_since_last".format(entity_scope, direction),
            "ML-01", "recency", entity_scope, direction, "all_past",
            "seconds_since_last", "{},timestamp".format(entity_scope),
            "last transaction uses past_timestamp < current_timestamp",
            "no previous transaction -> NaN",
            "medium", "recency", "selected",
            "first flag와 같이 사용", "{}_{}_recency".format(entity_scope, direction)
        )

        add_record(
            "flag__{}__{}__is_first_tx".format(entity_scope, direction),
            "ML-01", "flag", entity_scope, direction, "all_past",
            "is_first", "{},timestamp".format(entity_scope),
            "true only when no past transaction exists",
            "first -> 1 else 0",
            "low", "cold start", "selected",
            "recency NaN 해석 보조", "{}_{}_recency".format(entity_scope, direction)
        )


    # baseline P0 범위 아님. TODO로만 기록
    add_record(
        "cur_vs_mean_ratio",
        "TODO",
        "ratio",
        "transaction_vs_history",
        "current",
        "future",
        "current_amount_div_past_mean",
        "amount, history_amount_mean",
        "must use past_timestamp < current_timestamp",
        "zero denominator -> define epsilon or 0 before use",
        "medium",
        "amount anomaly",
        "todo",
        "TODO: Stage 0 확장 후보 cur_vs_mean_ratio",
        "amount_ratio_extension",
    )

    add_record(
        "pair_tgap",
        "TODO",
        "pair",
        "sender_receiver",
        "forward",
        "future",
        "seconds_since_last_pair_tx",
        "sender_account, receiver_account, timestamp",
        "must use past_timestamp < current_timestamp",
        "no previous pair -> NaN with first flag",
        "medium",
        "pair recency",
        "todo",
        "TODO: Pair 확장 후보 pair tgap, is_new_pair",
        "pair_extension",
    )

    add_record(
        "is_new_pair",
        "TODO",
        "pair",
        "sender_receiver",
        "forward",
        "future",
        "is_first_pair",
        "sender_account, receiver_account, timestamp",
        "must use past_timestamp < current_timestamp",
        "first pair -> 1 else 0",
        "low",
        "new counterparty pair",
        "todo",
        "TODO: Pair 확장 후보 pair tgap, is_new_pair",
        "pair_extension",
    )

    cols = [
        "feature_name",
        "stage",
        "feature_family",
        "entity_scope",
        "direction",
        "window",
        "aggregation_function",
        "input_columns",
        "leakage_rule",
        "null_policy",
        "computational_cost",
        "aml_typology",
        "selection_status",
        "notes",
        "correlation_group",
    ]

    return pd.DataFrame(records)[cols]


# feature 설명과 누수 방지 기준을 catalog로 저장합니다.
feature_catalog = make_feature_catalog()
feature_catalog_path = CONFIG["OUTPUT_DIR"] / "feature_catalog.csv"
feature_catalog.to_csv(feature_catalog_path, index=False, encoding="utf-8-sig")

print("saved:", feature_catalog_path)
feature_catalog.head(10)


saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\feature_catalog.csv


,feature_name,stage,feature_family,entity_scope,direction,window,aggregation_function,input_columns,leakage_rule,null_policy,computational_cost,aml_typology,selection_status,notes,correlation_group
0,amount__current__value,ML-00,amount,transaction,current,none,identity,amount,current row only,source amount must be non-null,low,amount anomaly,selected,,
1,amount__current__log1p,ML-00,amount,transaction,current,none,log1p,amount,current row only,negative amount -> null,low,amount anomaly,selected,raw amount와 같이 실험 후 비교,amount_scale
2,cat__from_bank__code,ML-00,categorical,transaction,current,none,train_only_code,from_bank,mapping is fitted on train split only,unknown in val/test -> -1,low,institution/payment/currency pattern,selected,정수 코드 자체에 순서 의미는 없음,categorical_from_bank
3,cat__to_bank__code,ML-00,categorical,transaction,current,none,train_only_code,to_bank,mapping is fitted on train split only,unknown in val/test -> -1,low,institution/payment/currency pattern,selected,정수 코드 자체에 순서 의미는 없음,categorical_to_bank
4,cat__payment_currency__code,ML-00,categorical,transaction,current,none,train_only_code,payment_currency,mapping is fitted on train split only,unknown in val/test -> -1,low,institution/payment/currency pattern,selected,정수 코드 자체에 순서 의미는 없음,categorical_payment_currency
5,cat__receiving_currency__code,ML-00,categorical,transaction,current,none,train_only_code,receiving_currency,mapping is fitted on train split only,unknown in val/test -> -1,low,institution/payment/currency pattern,selected,정수 코드 자체에 순서 의미는 없음,categorical_receiving_currency
6,cat__payment_format__code,ML-00,categorical,transaction,current,none,train_only_code,payment_format,mapping is fitted on train split only,unknown in val/test -> -1,low,institution/payment/currency pattern,selected,정수 코드 자체에 순서 의미는 없음,categorical_payment_format
7,time__row__hour,ML-00,time,transaction,current,none,hour,timestamp,derived from current timestamp only,timestamp valid -> not null,low,temporal pattern,selected,transaction hour,time_current
8,time__row__dayofweek,ML-00,time,transaction,current,none,dayofweek,timestamp,derived from current timestamp only,timestamp valid -> not null,low,temporal pattern,selected,weekday,time_current
9,time__row__is_weekend,ML-00,time,transaction,current,none,is_weekend,timestamp,derived from current timestamp only,timestamp valid -> not null,low,temporal pattern,selected,weekend flag,time_current


## 14. baseline metrics template 생성
모델 결과를 같은 형식으로 기록할 수 있도록 템플릿을 만듭니다. 실제 성능 수치는 모델 학습 후 채우도록 구성했습니다.

In [ ]:
# ============================================================
# ml_feature_columns.csv
# ============================================================

# 컬럼명을 기준으로 feature 계열을 분류합니다.
def infer_feature_group(col):
    if col.startswith("amount__"):
        return "amount"
    if col.startswith("cat__"):
        return "categorical"
    if col.startswith("time__"):
        return "time"
    if col.startswith("timehist__"):
        return "history"
    if col.startswith("recency__"):
        return "recency"
    if col.startswith("flag__"):
        return "flag"
    return "unknown"


def infer_stage(col):
    if col in ML00_FEATURE_COLS:
        return "ML-00"
    if col in ML01_HISTORY_FEATURE_COLS:
        return "ML-01"
    return "unknown"


def infer_leakage_risk(col):
    if col.startswith("timehist__") or col.startswith("recency__") or col.startswith("flag__"):
        return "low_if_timestamp_lt_current_validated"
    if col.startswith("cat__"):
        return "low_if_train_only_mapping"
    if col.startswith("time__"):
        return "low_current_timestamp_only"
    return "low_current_row_only"


# 실제 모델 입력으로 쓸 컬럼과 제외할 메타 컬럼을 구분합니다.
def make_ml_feature_columns():
    all_cols = sorted(set(ML00_FEATURE_COLS + ML00_NO_TIME_FEATURE_COLS + ML01_FEATURE_COLS))
    records = []

    for col in all_cols:
        dtype = safe_schema_get(ml_exp01.schema, col, safe_schema_get(ml_exp00.schema, col, "unknown"))
        records.append({
            "column_name": col,
            "used_in_ml": True,
            "feature_group": infer_feature_group(col),
            "stage": infer_stage(col),
            "dtype": dtype,
            "excluded_reason": "",
            "leakage_risk": infer_leakage_risk(col),
        })

    excluded_cols = [
        "tx_id", "timestamp", "split", "label",
        "sender_account", "receiver_account",
        "from_bank", "to_bank",
        "payment_currency", "receiving_currency", "payment_format",
    ]

    for col in excluded_cols:
        if col in df_ml01_base.columns:
            records.append({
                "column_name": col,
                "used_in_ml": False,
                "feature_group": "meta_or_raw",
                "stage": "excluded",
                "dtype": safe_schema_get(df_ml01_base.schema, col),
                "excluded_reason": "model input에서는 제외",
                "leakage_risk": "excluded_from_model_input",
            })

    return pd.DataFrame(records)


ml_feature_columns = make_ml_feature_columns()
ml_feature_columns_path = CONFIG["OUTPUT_DIR"] / "ml_feature_columns.csv"
ml_feature_columns.to_csv(ml_feature_columns_path, index=False, encoding="utf-8-sig")

print("saved:", ml_feature_columns_path)
ml_feature_columns.head(15)


saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_feature_columns.csv


,column_name,used_in_ml,feature_group,stage,dtype,excluded_reason,leakage_risk
0,amount__current__log1p,True,amount,ML-00,Float64,,low_current_row_only
1,amount__current__value,True,amount,ML-00,Float64,,low_current_row_only
2,cat__from_bank__code,True,categorical,ML-00,Int32,,low_if_train_only_mapping
3,cat__payment_currency__code,True,categorical,ML-00,Int32,,low_if_train_only_mapping
4,cat__payment_format__code,True,categorical,ML-00,Int32,,low_if_train_only_mapping
5,cat__receiving_currency__code,True,categorical,ML-00,Int32,,low_if_train_only_mapping
6,cat__to_bank__code,True,categorical,ML-00,Int32,,low_if_train_only_mapping
7,flag__receiver__in__is_first_tx,True,flag,ML-01,Int8,,low_if_timestamp_lt_current_validated
8,flag__sender__out__is_first_tx,True,flag,ML-01,Int8,,low_if_timestamp_lt_current_validated
9,recency__receiver__in__seconds_since_last,True,recency,ML-01,Float64,,low_if_timestamp_lt_current_validated


## 15. 최종 산출물 확인
생성된 파일 목록과 주요 요약을 출력합니다. 실행 후 이 셀만 봐도 정상 완료 여부를 빠르게 확인할 수 있습니다.

In [ ]:
# 이 셀은 저장 전 leakage check를 모아서 실행합니다.
# ============================================================
# leakage check
# P0 실패 시 파일 저장 중단
# ============================================================

LEAKAGE_RESULTS = []

# leakage check 결과를 같은 형식으로 모읍니다.
def add_check_result(check_name, status, severity, details, n_failed_rows=0):
    LEAKAGE_RESULTS.append({
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "details": details,
        "n_failed_rows": int(n_failed_rows),
    })

    mark = "PASS" if status == "PASS" else "FAIL"
    print("[{}] [{}] {} | failed={} | {}".format(
        mark, severity, check_name, n_failed_rows, details
    ))


def _get_split_ts(summary_pd, split_name, col):
    sub = summary_pd.loc[summary_pd["split"] == split_name, col]
    if sub.empty:
        raise ValueError("split '{}' 가 비어 있습니다.".format(split_name))
    return pd.to_datetime(sub.iloc[0])


def validate_time_split(df):
    s = make_split_summary(df).to_pandas()

    train_max = _get_split_ts(s, "train", "timestamp_max")
    val_min = _get_split_ts(s, "val", "timestamp_min")
    val_max = _get_split_ts(s, "val", "timestamp_max")
    test_min = _get_split_ts(s, "test", "timestamp_min")

    ok = train_max <= val_min and val_max <= test_min

    add_check_result(
        "validate_time_split",
        "PASS" if ok else "FAIL",
        "P0",
        "train_max={}, val_min={}, val_max={}, test_min={}".format(
            train_max, val_min, val_max, test_min
        ),
        0 if ok else df.height,
    )


def validate_no_timestamp_overlap_between_splits(df):
    overlap = (
        df.group_by("timestamp")
        .agg(pl.col("split").n_unique().alias("n_splits"))
        .filter(pl.col("n_splits") > 1)
    )

    add_check_result(
        "validate_no_timestamp_overlap_between_splits",
        "PASS" if overlap.height == 0 else "FAIL",
        "P0",
        "same timestamp must not appear in multiple splits",
        overlap.height,
    )


def validate_no_forbidden_ml_columns(feature_cols, dataset_name):
    exact_forbidden = {x.lower() for x in CONFIG["EXACT_FORBIDDEN_MODEL_COLUMNS"]}
    substring_forbidden = {x.lower() for x in CONFIG["SUBSTRING_FORBIDDEN_MODEL_TOKENS"]}

    failed = []

    for col in feature_cols:
        lower_col = col.lower()

        # 파생 시간 피처와 recency 피처는 허용
        # 원본 timestamp 컬럼 자체는 exact forbidden에서 차단
        if col.startswith("time__row__") or col.startswith("recency__"):
            continue

        if lower_col in exact_forbidden:
            failed.append(col)
            continue

        for token in substring_forbidden:
            if token in lower_col:
                failed.append(col)
                break

    failed = sorted(set(failed))

    add_check_result(
        "validate_no_forbidden_ml_columns__{}".format(dataset_name),
        "PASS" if len(failed) == 0 else "FAIL",
        "P0",
        "forbidden columns: {}".format(failed[:20]),
        len(failed),
    )


# 샘플 거래를 직접 재계산해서 history feature 누수를 확인합니다.
def direct_recompute_history_for_row(
    df,
    tx_id,
    entity_col,
    direction_label,
    entity_scope_label,
    window_name,
    window_str,
):
    row = df.filter(pl.col("tx_id") == tx_id)

    if row.height != 1:
        raise ValueError("tx_id row count must be 1: {}".format(tx_id))

    current_ts = row["timestamp"][0]
    entity_value = row[entity_col][0]
    lower_ts = current_ts - pd.Timedelta(window_str)

    past = df.filter(
        (pl.col(entity_col) == entity_value)
        & (pl.col("timestamp") >= lower_ts)
        & (pl.col("timestamp") < current_ts)
    )

    count_val = past.height
    sum_val = past["amount"].sum() if count_val > 0 else 0.0
    mean_val = sum_val / count_val if count_val > 0 else 0.0

    return {
        "timehist__{}__{}__tx_count__count__{}".format(
            entity_scope_label, direction_label, window_name
        ): count_val,
        "timehist__{}__{}__amount__sum__{}".format(
            entity_scope_label, direction_label, window_name
        ): float(sum_val),
        "timehist__{}__{}__amount__mean__{}".format(
            entity_scope_label, direction_label, window_name
        ): float(mean_val),
    }


# history feature가 현재/미래 거래를 참조하지 않는지 검증합니다.
def validate_no_current_or_future_history(df_with_features):
    n = df_with_features.height
    validation_mode = CONFIG.get("VALIDATION_MODE", "sample")

    if validation_mode not in ["sample", "full"]:
        raise ValueError("VALIDATION_MODE는 'sample' 또는 'full'만 허용합니다.")

    if validation_mode == "full":
        sample_df = df_with_features
        validation_rows = n
    else:
        validation_rows = min(CONFIG["MAX_VALIDATION_ROWS"], n)
        sample_df = df_with_features.sample(
            n=validation_rows,
            seed=CONFIG["RANDOM_SEED"],
            shuffle=True,
        )

    sample_tx_ids = sample_df["tx_id"].to_list()
    failed = []

    checks = []
    for w_name, w_str in windows.items():
        checks.append(("sender_account", "out", "sender", w_name, w_str))
        checks.append(("receiver_account", "in", "receiver", w_name, w_str))

    for tx_id in sample_tx_ids:
        for entity_col, direction_label, entity_scope_label, w_name, w_str in checks:
            recomputed = direct_recompute_history_for_row(
                df=df_with_features,
                tx_id=tx_id,
                entity_col=entity_col,
                direction_label=direction_label,
                entity_scope_label=entity_scope_label,
                window_name=w_name,
                window_str=w_str,
            )

            row = df_with_features.filter(pl.col("tx_id") == tx_id)

            for col_name, expected_val in recomputed.items():
                actual_val = row[col_name][0]

                if "mean" in col_name or "sum" in col_name:
                    # 금액 합계/평균은 float 저장 방식 때문에 아주 작은 오차가 생길 수 있음
                    ok = np.isclose(
                        float(actual_val),
                        float(expected_val),
                        rtol=1e-6,
                        atol=1e-4,
                    )
                else:
                    ok = int(actual_val) == int(expected_val)

                if not ok:
                    failed.append({
                        "tx_id": tx_id,
                        "feature": col_name,
                        "actual": actual_val,
                        "expected": expected_val,
                        "diff": float(actual_val) - float(expected_val),
                        "entity_col": entity_col,
                        "direction": direction_label,
                        "window": w_name,
                    })

                    if len(failed) >= 20:
                        break

            if len(failed) >= 20:
                break

        if len(failed) >= 20:
            break

    if len(failed) > 0:
        fail_path = CONFIG["OUTPUT_DIR"] / "leakage_history_failure_examples.csv"
        pd.DataFrame(failed).to_csv(fail_path, index=False, encoding="utf-8-sig")
    else:
        fail_path = None

    add_check_result(
        "validate_no_current_or_future_history",
        "PASS" if len(failed) == 0 else "FAIL",
        "P0",
        "validation_mode={}, validation_rows={}, total_rows={}, first_failures={}, failure_examples_path={}".format(
            validation_mode,
            validation_rows,
            n,
            failed[:5],
            fail_path,
        ),
        len(failed),
    )


def validate_first_sender_history_zero(df_with_features):
    first_col = "flag__sender__out__is_first_tx"

    if first_col not in df_with_features.columns:
        add_check_result(
            "validate_first_sender_history_zero",
            "FAIL",
            "P0",
            "missing column: {}".format(first_col),
            1,
        )
        return

    target_cols = [c for c in df_with_features.columns if c.startswith("timehist__sender__out__")]
    first_rows = df_with_features.filter(pl.col(first_col) == 1)

    failed_count = 0
    for c in target_cols:
        failed_count += first_rows.filter(pl.col(c).fill_null(0) != 0).height

    add_check_result(
        "validate_first_sender_history_zero",
        "PASS" if failed_count == 0 else "FAIL",
        "P0",
        "first sender rows should have zero sender history",
        failed_count,
    )


def validate_first_receiver_history_zero(df_with_features):
    first_col = "flag__receiver__in__is_first_tx"

    if first_col not in df_with_features.columns:
        add_check_result(
            "validate_first_receiver_history_zero",
            "FAIL",
            "P0",
            "missing column: {}".format(first_col),
            1,
        )
        return

    target_cols = [c for c in df_with_features.columns if c.startswith("timehist__receiver__in__")]
    first_rows = df_with_features.filter(pl.col(first_col) == 1)

    failed_count = 0
    for c in target_cols:
        failed_count += first_rows.filter(pl.col(c).fill_null(0) != 0).height

    add_check_result(
        "validate_first_receiver_history_zero",
        "PASS" if failed_count == 0 else "FAIL",
        "P0",
        "first receiver rows should have zero receiver history",
        failed_count,
    )


# leakage 검증 범위가 sample인지 full인지 리포트에 남깁니다.
def validate_history_validation_scope(df_with_features):
    n = df_with_features.height
    validation_mode = CONFIG.get("VALIDATION_MODE", "sample")

    if validation_mode == "full":
        validation_rows = n
        details = "validation_mode=full, validation_rows={}, total_rows={}".format(
            validation_rows,
            n,
        )
        severity = "INFO"
    elif validation_mode == "sample":
        validation_rows = min(CONFIG["MAX_VALIDATION_ROWS"], n)
        details = "validation_mode=sample, validation_rows={}, total_rows={}, MAX_VALIDATION_ROWS={}".format(
            validation_rows,
            n,
            CONFIG["MAX_VALIDATION_ROWS"],
        )
        severity = "P1"
    else:
        add_check_result(
            "validate_history_validation_scope",
            "FAIL",
            "P0",
            "invalid VALIDATION_MODE={}".format(validation_mode),
            1,
        )
        return

    add_check_result(
        "validate_history_validation_scope",
        "PASS",
        severity,
        details,
        0,
    )


def run_all_ml_validations():
    global LEAKAGE_RESULTS
    LEAKAGE_RESULTS = []

    validate_history_validation_scope(df_ml01_base)

    validate_time_split(df)
    validate_no_timestamp_overlap_between_splits(df)

    validate_no_forbidden_ml_columns(ML00_FEATURE_COLS, "ml_exp00")
    validate_no_forbidden_ml_columns(ML00_NO_TIME_FEATURE_COLS, "ml_exp00_no_time")
    validate_no_forbidden_ml_columns(ML01_FEATURE_COLS, "ml_exp01")

    validate_no_current_or_future_history(df_ml01_base)
    validate_first_sender_history_zero(df_ml01_base)
    validate_first_receiver_history_zero(df_ml01_base)

    leakage_df = pd.DataFrame(LEAKAGE_RESULTS)
    leakage_path = CONFIG["OUTPUT_DIR"] / "leakage_check.csv"
    leakage_df.to_csv(leakage_path, index=False, encoding="utf-8-sig")
    print("saved:", leakage_path)

    p0_fail = leakage_df[
        (leakage_df["severity"] == "P0")
        & (leakage_df["status"] != "PASS")
    ]

    if len(p0_fail) > 0:
        display(p0_fail)
        raise AssertionError("P0 leakage check failed. 파일 저장을 중단합니다.")

    return leakage_df


leakage_check = run_all_ml_validations()
leakage_check


[PASS] [P1] validate_history_validation_scope | failed=0 | validation_mode=sample, validation_rows=5000, total_rows=5078345, MAX_VALIDATION_ROWS=5000
[PASS] [P0] validate_time_split | failed=0 | train_max=2022-09-06 13:35:00, val_min=2022-09-06 13:36:00, val_max=2022-09-08 16:12:00, test_min=2022-09-08 16:13:00
[PASS] [P0] validate_no_timestamp_overlap_between_splits | failed=0 | same timestamp must not appear in multiple splits
[PASS] [P0] validate_no_forbidden_ml_columns__ml_exp00 | failed=0 | forbidden columns: []
[PASS] [P0] validate_no_forbidden_ml_columns__ml_exp00_no_time | failed=0 | forbidden columns: []
[PASS] [P0] validate_no_forbidden_ml_columns__ml_exp01 | failed=0 | forbidden columns: []
[PASS] [P0] validate_no_current_or_future_history | failed=0 | validation_mode=sample, validation_rows=5000, total_rows=5078345, first_failures=[], failure_examples_path=None
[PASS] [P0] validate_first_sender_history_zero | failed=0 | first sender rows should have zero sender history
[PAS

,check_name,status,severity,details,n_failed_rows
0,validate_history_validation_scope,PASS,P1,"validation_mode=sample, validation_rows=5000, total_rows=5078345, MAX_VALIDATION_ROWS=5000",0
1,validate_time_split,PASS,P0,"train_max=2022-09-06 13:35:00, val_min=2022-09-06 13:36:00, val_max=2022-09-08 16:12:00, test_min=2022-09-08 16:13:00",0
2,validate_no_timestamp_overlap_between_splits,PASS,P0,same timestamp must not appear in multiple splits,0
3,validate_no_forbidden_ml_columns__ml_exp00,PASS,P0,forbidden columns: [],0
4,validate_no_forbidden_ml_columns__ml_exp00_no_time,PASS,P0,forbidden columns: [],0
5,validate_no_forbidden_ml_columns__ml_exp01,PASS,P0,forbidden columns: [],0
6,validate_no_current_or_future_history,PASS,P0,"validation_mode=sample, validation_rows=5000, total_rows=5078345, first_failures=[], failure_examples_path=None",0
7,validate_first_sender_history_zero,PASS,P0,first sender rows should have zero sender history,0
8,validate_first_receiver_history_zero,PASS,P0,first receiver rows should have zero receiver history,0


In [ ]:
# ============================================================
# baseline metrics template
# ============================================================

# 추후 baseline metric을 채울 수 있도록 템플릿 행을 만듭니다.
def metric_row(experiment_name, split_name, data, notes):
    part = data.filter(pl.col("split") == split_name)

    return {
        "experiment_name": experiment_name,
        "split": split_name,
        "model_name": "",
        "n_rows": int(part.height),
        "positive_ratio": float(part["label"].mean()) if part.height > 0 else np.nan,
        "roc_auc": "",
        "pr_auc": "",
        "f1": "",
        "precision": "",
        "recall": "",
        "threshold": "",
        "notes": notes,
    }


baseline_metrics_template = pd.DataFrame([
    metric_row("ml_exp00", "train", ml_exp00, "baseline current-row features"),
    metric_row("ml_exp00", "val", ml_exp00, "validation set"),
    metric_row("ml_exp00", "test", ml_exp00, "final evaluation only"),
    metric_row("ml_exp00_no_time", "train", ml_exp00_no_time, "baseline without time features"),
    metric_row("ml_exp00_no_time", "val", ml_exp00_no_time, "validation set"),
    metric_row("ml_exp00_no_time", "test", ml_exp00_no_time, "final evaluation only"),
    metric_row("ml_exp01", "train", ml_exp01, "ML-00 + Stage 0 history"),
    metric_row("ml_exp01", "val", ml_exp01, "validation set"),
    metric_row("ml_exp01", "test", ml_exp01, "final evaluation only"),
])

metrics_template_path = CONFIG["OUTPUT_DIR"] / "ml_baseline_metrics_template.csv"
baseline_metrics_template.to_csv(metrics_template_path, index=False, encoding="utf-8-sig")

print("saved:", metrics_template_path)
baseline_metrics_template


saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_baseline_metrics_template.csv


,experiment_name,split,model_name,n_rows,positive_ratio,roc_auc,pr_auc,f1,precision,recall,threshold,notes
0,ml_exp00,train,,3046861,0.000754,,,,,,,baseline current-row features
1,ml_exp00,val,,1015920,0.001066,,,,,,,validation set
2,ml_exp00,test,,1015564,0.001769,,,,,,,final evaluation only
3,ml_exp00_no_time,train,,3046861,0.000754,,,,,,,baseline without time features
4,ml_exp00_no_time,val,,1015920,0.001066,,,,,,,validation set
5,ml_exp00_no_time,test,,1015564,0.001769,,,,,,,final evaluation only
6,ml_exp01,train,,3046861,0.000754,,,,,,,ML-00 + Stage 0 history
7,ml_exp01,val,,1015920,0.001066,,,,,,,validation set
8,ml_exp01,test,,1015564,0.001769,,,,,,,final evaluation only


In [ ]:
# ============================================================
# parquet 저장
# 검증 통과 후 저장한다.
# ============================================================

def save_parquet(df, filename):
    path = CONFIG["OUTPUT_DIR"] / filename
    df.write_parquet(path)
    print("saved:", path, "| rows=", df.height, "| cols=", len(df.columns))
    return path


save_parquet(ml_exp00, "ml_exp00.parquet")
save_parquet(ml_exp00_no_time, "ml_exp00_no_time.parquet")
save_parquet(ml_exp01, "ml_exp01.parquet")

save_parquet(ml_exp00.filter(pl.col("split") == "train"), "ml_exp00_train.parquet")
save_parquet(ml_exp00.filter(pl.col("split") == "val"), "ml_exp00_val.parquet")
save_parquet(ml_exp00.filter(pl.col("split") == "test"), "ml_exp00_test.parquet")

save_parquet(ml_exp00_no_time.filter(pl.col("split") == "train"), "ml_exp00_no_time_train.parquet")
save_parquet(ml_exp00_no_time.filter(pl.col("split") == "val"), "ml_exp00_no_time_val.parquet")
save_parquet(ml_exp00_no_time.filter(pl.col("split") == "test"), "ml_exp00_no_time_test.parquet")

save_parquet(ml_exp01.filter(pl.col("split") == "train"), "ml_exp01_train.parquet")
save_parquet(ml_exp01.filter(pl.col("split") == "val"), "ml_exp01_val.parquet")
save_parquet(ml_exp01.filter(pl.col("split") == "test"), "ml_exp01_test.parquet")

# X/y 파일: label은 y 분리용, split은 분할 유지용이다.
save_parquet(ml_exp00_Xy, "ml_exp00_Xy.parquet")
save_parquet(ml_exp00_no_time_Xy, "ml_exp00_no_time_Xy.parquet")
save_parquet(ml_exp01_Xy, "ml_exp01_Xy.parquet")

save_parquet(ml_exp00_Xy.filter(pl.col("split") == "train"), "ml_exp00_Xy_train.parquet")
save_parquet(ml_exp00_Xy.filter(pl.col("split") == "val"), "ml_exp00_Xy_val.parquet")
save_parquet(ml_exp00_Xy.filter(pl.col("split") == "test"), "ml_exp00_Xy_test.parquet")

save_parquet(ml_exp00_no_time_Xy.filter(pl.col("split") == "train"), "ml_exp00_no_time_Xy_train.parquet")
save_parquet(ml_exp00_no_time_Xy.filter(pl.col("split") == "val"), "ml_exp00_no_time_Xy_val.parquet")
save_parquet(ml_exp00_no_time_Xy.filter(pl.col("split") == "test"), "ml_exp00_no_time_Xy_test.parquet")

save_parquet(ml_exp01_Xy.filter(pl.col("split") == "train"), "ml_exp01_Xy_train.parquet")
save_parquet(ml_exp01_Xy.filter(pl.col("split") == "val"), "ml_exp01_Xy_val.parquet")
save_parquet(ml_exp01_Xy.filter(pl.col("split") == "test"), "ml_exp01_Xy_test.parquet")

record_memory("parquet_saved", ml_exp01)


saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00.parquet | rows= 5078345 | cols= 16
saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_no_time.parquet | rows= 5078345 | cols= 13
saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp01.parquet | rows= 5078345 | cols= 50
saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_train.parquet | rows= 3046861 | cols= 16
saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_val.parquet | rows= 1015920 | cols= 16
saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_test.parquet | rows= 1015564 | cols= 16
saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_no_time_train.parquet | rows= 3046861 | cols= 13
saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_no_time_val.parquet | rows= 1015920 | cols= 13
saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_no_time_test.parquet | rows= 1015564 | cols= 13
saved: C:\Users\gusq

In [ ]:
# ============================================================
# 마지막 확인
# ============================================================

save_memory_profile()

expected_outputs = [
    "ml_exp00.parquet",
    "ml_exp00_no_time.parquet",
    "ml_exp01.parquet",

    "ml_exp00_train.parquet",
    "ml_exp00_val.parquet",
    "ml_exp00_test.parquet",

    "ml_exp00_no_time_train.parquet",
    "ml_exp00_no_time_val.parquet",
    "ml_exp00_no_time_test.parquet",

    "ml_exp01_train.parquet",
    "ml_exp01_val.parquet",
    "ml_exp01_test.parquet",

    "ml_exp00_Xy.parquet",
    "ml_exp00_no_time_Xy.parquet",
    "ml_exp01_Xy.parquet",

    "ml_exp00_Xy_train.parquet",
    "ml_exp00_Xy_val.parquet",
    "ml_exp00_Xy_test.parquet",

    "ml_exp00_no_time_Xy_train.parquet",
    "ml_exp00_no_time_Xy_val.parquet",
    "ml_exp00_no_time_Xy_test.parquet",

    "ml_exp01_Xy_train.parquet",
    "ml_exp01_Xy_val.parquet",
    "ml_exp01_Xy_test.parquet",

    "ml_feature_columns.csv",
    "split_summary.csv",
    "leakage_check.csv",
    "feature_catalog.csv",
    "ml_baseline_metrics_template.csv",
    "memory_profile.csv",
    "category_mapping_train_only.csv",
]

check_records = []

for fname in expected_outputs:
    path = CONFIG["OUTPUT_DIR"] / fname
    check_records.append({
        "file_name": fname,
        "exists": path.exists(),
        "path": str(path),
        "size_mb": path.stat().st_size / 1024 / 1024 if path.exists() else None,
    })

output_check = pd.DataFrame(check_records)
display(output_check)

missing = output_check[~output_check["exists"]]
if len(missing) > 0:
    raise FileNotFoundError("생성되지 않은 파일이 있습니다: {}".format(missing["file_name"].tolist()))

print("완료")
print("output:", CONFIG["OUTPUT_DIR"])


print("\nTODO")
print("TODO: Stage 0 확장 후보 cur_vs_mean_ratio")
print("TODO: Pair 확장 후보 pair tgap, is_new_pair")


saved: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\memory_profile.csv


,file_name,exists,path,size_mb
0,ml_exp00.parquet,True,C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00.parquet,124.879486
1,ml_exp00_no_time.parquet,True,C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_no_time.parquet,124.862556
2,ml_exp01.parquet,True,C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp01.parquet,599.370615
3,ml_exp00_train.parquet,True,C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_train.parquet,75.658406
4,ml_exp00_val.parquet,True,C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_val.parquet,23.612914
5,ml_exp00_test.parquet,True,C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_test.parquet,24.616023
6,ml_exp00_no_time_train.parquet,True,C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_no_time_train.parquet,76.144016
7,ml_exp00_no_time_val.parquet,True,C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_no_time_val.parquet,23.580467
8,ml_exp00_no_time_test.parquet,True,C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp00_no_time_test.parquet,24.447324
9,ml_exp01_train.parquet,True,C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery\ml_exp01_train.parquet,343.881455


완료
output: C:\Users\gusqh\Downloads\GraphAML\data\ml_delivery

TODO
TODO: Stage 0 확장 후보 cur_vs_mean_ratio
TODO: Pair 확장 후보 pair tgap, is_new_pair
